# Representation-level dissociation test (ACML revision)

Answers reviewers **R1.2 / R2.1 / R3.1**: the submitted study varied only last-layer heads, so
"not a representation problem" was read as unsupported. Here the **backbone itself** is fine-tuned
end-to-end under ERM / GroupDRO / group-balanced objectives, and the full head x calibration
comparison is re-run on each resulting representation.

**The verdict is falsifiable.** Two levers are compared head to head:

* *representation lever* -- the best worst-group coverage reachable under **marginal** calibration, maximising over representations and heads
* *calibration lever* -- the worst worst-group coverage reached under **Mondrian**, minimising over representations and heads

If the worst calibration cell still beats the best representation cell (CI-separated), the title
stands. If a robust representation closes the marginal gap on its own, the title must narrow.
Both outcomes are reported.

**Runtime (L4).** Waterbirds ~45 min total. CelebA ~2.5-3 h. Every fine-tune checkpoints to Drive
after each epoch, so a session that hits its limit **resumes** rather than restarting -- just
re-run the cell. Fine-tuned features are cached to Drive too, so a second session is nearly free.

Run Waterbirds -> **STOP and review** -> then CelebA.

## 0. Parameters -- **EDIT THESE**

In [ ]:
REPO_URL      = "https://github.com/octadion/vgscp"
REPO_BRANCH   = "main"
REPO_SOURCE   = "git"          # "git" | "drive"
DRIVE_CACHE   = "/content/drive/MyDrive/vgscp_cache"
REPO_DRIVE_ZIP= "/content/drive/MyDrive/vgscp.zip"
WATERBIRDS_URL= "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CELEBA_SOURCE = "kaggle"       # "kaggle" (needs kaggle.json) | "drive" | "skip"
CELEBA_DRIVE  = ""

# --- fine-tune budget (L4 defaults) -------------------------------------------------
FT_OBJECTIVES = ("erm", "groupdro", "reweight")
FT_SEEDS      = (0, 1, 2)      # clustering unit for every CI (R2.3)
WB_EPOCHS     = 10             # Waterbirds: 4,795 imgs -> ~3-5 min/run on L4
CELEBA_EPOCHS = 5              # CelebA: 162,770 imgs -> ~25-35 min/run on L4
CELEBA_MAX_TRAIN = None        # set e.g. 50000 to cut CelebA cost ~3x (documented subsample)
BATCH_SIZE    = 128
NUM_WORKERS   = 8              # decode is the bottleneck on CelebA, not the GPU
LR            = 1e-3
N_SPLITS      = 10
HEADS         = ("erm", "dfr", "groupdro_ll")
SCORES        = ("APS", "RAPS", "THR")

## 1. Drive + repo

In [ ]:
import os, sys, time, subprocess
def sh(c): print(subprocess.run(c, shell=True, capture_output=True, text=True).stdout[-2000:])

from google.colab import drive
drive.mount("/content/drive"); os.makedirs(DRIVE_CACHE, exist_ok=True)

REPO_DIR = "/content/vgscp"
if REPO_SOURCE == "git":
    sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh(f"rm -rf {REPO_DIR} && mkdir -p {REPO_DIR} && unzip -q {REPO_DRIVE_ZIP} -d {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

import torch
print("repo:", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (fix runtime!)")

## 2. Drive-backed caches

All four are symlinked to Drive. `cache_finetune` and `study` matter most: the first makes a re-run
nearly free, the second means the results CSV survives the session. *(The original grid notebook
Drive-backed only the feature caches, which is why `grid_records.csv` did not persist.)*

In [ ]:
for c in ("cache_clip", "cache_resnet", "cache_finetune", "study"):
    sh(f"rm -rf results/{c}"); os.makedirs(f"{DRIVE_CACHE}/{c}", exist_ok=True)
    os.makedirs("results", exist_ok=True); sh(f"ln -s {DRIVE_CACHE}/{c} results/{c}")
os.makedirs(f"{DRIVE_CACHE}/cache_finetune/ckpt", exist_ok=True)   # per-epoch resume lives here
sh("ls -la results/")

## 3. Datasets

In [ ]:
from study_robust_train.colab_data import prepare_waterbirds, prepare_celeba
os.environ["WATERBIRDS_ROOT"] = prepare_waterbirds(DRIVE_CACHE, WATERBIRDS_URL)
CELEBA_ROOT = prepare_celeba(DRIVE_CACHE, source=CELEBA_SOURCE, celeba_drive=CELEBA_DRIVE)
CELEBA_OK = bool(CELEBA_ROOT) and os.path.isdir(CELEBA_ROOT)
if CELEBA_OK: os.environ["CELEBA_ROOT"] = CELEBA_ROOT
print("WATERBIRDS_ROOT =", os.environ["WATERBIRDS_ROOT"], "| CelebA OK =", CELEBA_OK)

## 4. Sanity gate -- validate the analysis machinery before spending GPU time

In [ ]:
rc = subprocess.run([sys.executable, "-m", "study_robust_train.validate_representation"],
                    capture_output=True, text=True)
print(rc.stdout[-3000:])
assert rc.returncode == 0, "validator FAILED -- do not burn GPU time until this passes"

## 5. Waterbirds -- fine-tune 3 objectives x 3 seeds

~3-5 min per run on an L4, so ~45 min total including feature extraction. Re-running this cell
after an interruption resumes from the last completed epoch.

In [ ]:
from study_robust_train.representation import build_repr_griddata

def cfg_for(dataset, epochs, max_train=None):
    base = {"finetune": {"device": "cuda", "epochs": epochs, "lr": LR,
                         "batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS, "amp": True,
                         "max_train": max_train, "cache_dir": "results/cache_finetune",
                         "ckpt_dir": "results/cache_finetune/ckpt"}}
    if dataset == "waterbirds":
        base["dataset"] = {"root": os.environ["WATERBIRDS_ROOT"], "image_size": 224,
                           "n_classes": 2, "download": False}
    else:
        base["dataset"] = {"root": os.environ["CELEBA_ROOT"], "n_classes": 2}
    return base

def build_all(dataset, epochs, max_train=None):
    cfg, data, failed = cfg_for(dataset, epochs, max_train), {}, []
    for obj in FT_OBJECTIVES:
        for s in FT_SEEDS:
            t = time.time()
            try:
                data[(dataset, obj, s)] = build_repr_griddata(dataset, obj, cfg, ft_seed=s)
                print(f"[built] {dataset}/{obj}/s{s}  ({(time.time()-t)/60:.1f} min)", flush=True)
            except Exception as e:
                failed.append((dataset, obj, s, repr(e))); print(f"[FAIL] {dataset}/{obj}/s{s}: {e}")
    return data, failed

wb_data, wb_failed = build_all("waterbirds", WB_EPOCHS)
print("\nbuilt:", len(wb_data), "| failed:", wb_failed)

## 6. Run the comparison on Waterbirds

In [ ]:
from study_robust_train.representation import run_representation_experiment, write_representation_md
from study_robust_train.grid import write_csv
from IPython.display import Markdown, display

wb_out = run_representation_experiment(wb_data, heads=HEADS, scores=SCORES, n_splits=N_SPLITS)
write_csv(wb_out["records"], "results/study/representation_records.csv")
display(Markdown(write_representation_md(wb_out, "REPRESENTATION.md")))

## 7. STOP -- review before spending the CelebA budget

Read the two-lever verdict above. Either outcome is publishable, but they lead to different
revisions, so decide the framing now rather than after another 3 GPU-hours:

* **CALIBRATION LEVER DOMINATES** -> the title survives; CelebA becomes replication.
* **representation lever competitive** -> the title must narrow, and CelebA then matters *more*,
  because the claim becomes dataset-specific and needs a second dataset to characterise.

Also sanity-check that the fine-tune actually did something: `train-wg` in the logs above should
rise for `groupdro` / `reweight` relative to `erm`. If it did not, the robust objective never
changed the representation and the comparison is vacuous -- raise `groupdro_eta` or switch to
`optimizer="sgd"` before spending the CelebA budget.

## 8. CelebA (heavier -- ~2.5-3 h on L4; resumable)

In [ ]:
assert CELEBA_OK, "CelebA unavailable -- set CELEBA_SOURCE / upload kaggle.json"
cel_data, cel_failed = build_all("celeba", CELEBA_EPOCHS, CELEBA_MAX_TRAIN)
print("\nbuilt:", len(cel_data), "| failed:", cel_failed)

## 9. Combined report (Waterbirds + CelebA)

In [ ]:
all_data = {**wb_data, **cel_data}
out = run_representation_experiment(all_data, heads=HEADS, scores=SCORES, n_splits=N_SPLITS)
write_csv(out["records"], "results/study/representation_records.csv")
display(Markdown(write_representation_md(out, "REPRESENTATION.md")))
print("\nPersisted to Drive:", f"{DRIVE_CACHE}/study/representation_records.csv")

## 10. Pull the older CSVs down as well

The bucket-B reanalysis (TOST, cluster bootstrap, correlation CIs) needs the *original* grid
records. This lists what actually survived on Drive.

In [ ]:
sh(f"ls -la {DRIVE_CACHE}/study/")
for f in ("representation_records.csv", "grid_records.csv", "calibration_ablation.csv",
          "predicted_group_mondrian.csv"):
    p = f"{DRIVE_CACHE}/study/{f}"
    print(("FOUND   " if os.path.exists(p) else "MISSING ") + p)